# Building a synthetic version of a real customer table

This is the hands-on companion to Session 4's synthetic data lessons. Those lessons covered when synthetic data makes sense, how a model actually learns a table, and how to judge whether the result is good enough. This notebook builds one, on a real dataset, start to finish: load real data, preprocess it, train and tune CTGAN and TVAE, generate a synthetic version, and check whether it actually works for the thing we built it for.

**Time**: 45-90 minutes, most of it spent waiting on training, not writing code
**Cost**: $0 -- everything here runs locally or on Colab's own CPU/GPU. No API key, no billing, nothing metered.

> **Disclaimer.** This is educational guidance, not a production recipe. `auto_synthetic_data_platform` is not an official Google product and has had a single release since February 2024 -- this notebook includes the exact install steps needed to get it running on a current Python, verified end to end, but expect to revisit them if the underlying packages move again. A real production pipeline needs a real review by someone with the relevant expertise.

## The dataset

The analysis this session builds toward is the one named in the lessons: a propensity model predicting whether a customer buys again within 90 days. That needs a real dataset with real repeat-purchase behaviour in it, not an invented one.

This notebook uses the [Olist Brazilian e-commerce dataset](https://huggingface.co/datasets/miminmoons/olist-ecommerce-for-delivery-and-review-prediction): about 100,000 real, anonymised orders placed at a Brazilian online marketplace between September 2016 and September 2018. It's rich enough to build a genuine customer-level table from, and it's real, so every number in this notebook, including the uncomfortable ones, reflects actual purchase behaviour rather than something invented to make the example work.

Two of the columns the lessons used as illustration don't exist in Olist at all, and it's worth naming the substitution rather than quietly working around it:

- Olist has no age data of any kind, so **age band** is dropped. In its place, this notebook uses **top product category**, the category a customer bought from most, since it's a real column with genuine marketing value (it's exactly the kind of thing a personalisation model would use).
- Olist has no marketing-channel or acquisition-source data, so **channel of first visit** is replaced with **primary payment type** (credit card, boleto, voucher, debit card), the closest real categorical column available.

Everything else is exactly what the lessons describe: region (the customer's state), number of orders, average basket value, days since the last order, and the target, whether the customer bought again within 90 days.

### Avoiding a leak before it happens

The naive way to build this table, count each customer's total orders and check whether their *last* order was followed by another one, leaks the future into the present: a customer's own final order count already tells you whether they bought again. This notebook instead fixes a single cutoff date late enough that every customer has a full, uncensored 90-day window after it, and splits strictly on that date: every feature (region, order count, basket value, days since last order, payment type, category) is computed only from orders *before* the cutoff, and the label looks only at orders *after* it. That's the only way "bought again within 90 days" means what it says.

In [ ]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import shutil, os

REPO = "miminmoons/olist-ecommerce-for-delivery-and-review-prediction"
FILES = [
    "data/olist_customers_dataset.csv",
    "data/olist_orders_dataset.csv",
    "data/olist_order_items_dataset.csv",
    "data/olist_order_payments_dataset.csv",
    "data/olist_products_dataset.csv",
]

os.makedirs("data", exist_ok=True)
for f in FILES:
    path = hf_hub_download(repo_id=REPO, filename=f, repo_type="dataset")
    shutil.copy(path, os.path.join("data", os.path.basename(f)))

customers = pd.read_csv("data/olist_customers_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
items = pd.read_csv("data/olist_order_items_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
print("loaded:", len(customers), "customers,", len(orders), "orders")

In [ ]:
# Cancelled orders never delivered anything, so they carry no purchase signal.
orders = orders[orders["order_status"] != "canceled"]

# One payment value per order (payments can be split across installments/methods),
# plus whichever payment type carried the largest share of that order.
order_value = payments.groupby("order_id")["payment_value"].sum().rename("order_value")
order_payment_type = (
    payments.sort_values("payment_value", ascending=False)
    .groupby("order_id")["payment_type"].first()
    .rename("payment_type")
)

# The product category with the most items in each order.
items_with_cat = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
order_top_category = (
    items_with_cat.groupby("order_id")["product_category_name"]
    .agg(lambda s: s.value_counts().idxmax() if s.notna().any() else np.nan)
    .rename("top_category")
)

o = orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
o = o.merge(order_value, on="order_id", how="left")
o = o.merge(order_payment_type, on="order_id", how="left")
o = o.merge(order_top_category, on="order_id", how="left")
o = o.sort_values(["customer_unique_id", "order_purchase_timestamp"])

# Cutoff: 90 days before the last order in the dataset, so the forward window
# is never censored.
MAX_DATE = o["order_purchase_timestamp"].max()
CUTOFF = MAX_DATE - pd.Timedelta(days=90)
print("last order in the data:", MAX_DATE.date(), " |  cutoff:", CUTOFF.date())

history = o[o["order_purchase_timestamp"] < CUTOFF]
future = o[(o["order_purchase_timestamp"] >= CUTOFF) & (o["order_purchase_timestamp"] < CUTOFF + pd.Timedelta(days=90))]
repeat_customers = set(future["customer_unique_id"].unique())

def most_common(s):
    s = s.dropna()
    return s.value_counts().idxmax() if len(s) else np.nan

customer_table = history.groupby("customer_unique_id").agg(
    region=("customer_state", "first"),
    num_orders=("order_id", "nunique"),
    avg_basket_value=("order_value", "mean"),
    last_order_date=("order_purchase_timestamp", "max"),
    primary_payment_type=("payment_type", most_common),
    top_category=("top_category", most_common),
)
customer_table["days_since_last_order"] = (CUTOFF - customer_table["last_order_date"]).dt.days
customer_table["bought_again_90d"] = customer_table.index.isin(repeat_customers).astype(int)
customer_table = customer_table.drop(columns=["last_order_date"]).reset_index(drop=True)

print("\ncustomer table:", customer_table.shape)
customer_table.head()

### The real imbalance, before touching a model

Before doing anything else with this table, apply one of the checks this session's lessons will run formally in a moment: how rare is the thing we're trying to predict, really?

In [ ]:
positive_rate = customer_table["bought_again_90d"].mean()
print(f"customers: {len(customer_table)}")
print(f"bought again within 90 days: {customer_table['bought_again_90d'].sum()} ({positive_rate:.3%})")

That number, well under 1%, is what the platform's own class-imbalance check would call **extreme**. This isn't a flaw in how the table was built. It's what a real 90-day repeat-purchase rate on a real marketplace actually looks like, and it's exactly the scenario the imbalance thresholds in this session's lessons, and the rare-class evaluation lessons from Session 2, were written for.

It's also, honestly, too rare to train a CTGAN or TVAE model on quickly. With well under 1% of rows positive, a model can hit a very low loss while barely representing the minority class at all, and every training run in this notebook would take a very long time on a CPU. So the rest of this notebook works with a **stratified sample**: every customer who did buy again, plus a random sample of those who didn't, at a ratio chosen to bring the positive rate to roughly 6% (still a real, moderate imbalance by the same thresholds, just one a laptop can train on in a reasonable time). A production run would skip this step and tune against the full table.

In [ ]:
rng = np.random.RandomState(42)
positives = customer_table[customer_table["bought_again_90d"] == 1]
negatives = customer_table[customer_table["bought_again_90d"] == 0]
negative_sample = negatives.sample(n=len(positives) * 15, random_state=42)
working_table = pd.concat([positives, negative_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print("working table:", working_table.shape, " positive rate:", working_table["bought_again_90d"].mean())

## Installing the platform

This week's package is [Auto Synthetic Data Platform](https://github.com/google-marketing-solutions/auto_synthetic_data_platform), built by Google's EMEA gTech data science team on top of [synthcity](https://github.com/vanderschaarlab/synthcity). It is not an official Google product, and it has had exactly one release since February 2024 -- worth knowing before the very first cell, because it means `pip install auto_synthetic_data_platform` does not work on a current Python. Its one release pins `numpy==1.23.5`, `torch==1.13.1`, and several other exact versions that no longer resolve.

A second, independent problem sits inside `synthcity` itself: a plain `pip install synthcity` resolves a `torch` version that satisfies synthcity's own pin, but also pulls in a version of `opacus` (a differential-privacy dependency used internally) that needs a newer `torch` than the one just installed. Loading the library crashes on import, before any code runs, let alone this notebook's code.

The recipe below works around both problems: install `synthcity` unpinned so modern dependencies resolve, install the platform straight from its GitHub source with its pins relaxed, and pin `opacus` down to a version that predates the conflicting requirement. Verified end to end with a real training run while writing this notebook.

In [ ]:
%%bash
pip install -q synthcity
git clone -q https://github.com/google-marketing-solutions/auto_synthetic_data_platform.git
sed -i.bak 's/ == / >= /' auto_synthetic_data_platform/requirements.txt
pip install -q --no-deps -e ./auto_synthetic_data_platform
pip install -q "opacus<1.5" ipython kaleido fastapi uvicorn python-multipart plotly
echo "install complete

In [ ]:
from auto_synthetic_data_platform import preprocessing
from auto_synthetic_data_platform import numerical_variables_processing
from auto_synthetic_data_platform import synthetic_data_model_tuning as sdmt
from synthcity.plugins import Plugins
from synthcity.plugins.core import dataloader
import pathlib

print("import ok")

## Declaring column types

The platform's `Preprocessor` needs to be told, explicitly, which columns are numerical and which are categorical. It will not infer this, and it shouldn't have to: `num_orders`, `avg_basket_value`, and `days_since_last_order` are genuinely numerical, quantities where the distance between two values means something. `region`, `primary_payment_type`, `top_category`, and the target itself are categorical labels, even though `bought_again_90d` happens to be stored as 0 and 1.

In [ ]:
column_metadata = {
    "categorical": ["region", "primary_payment_type", "top_category", "bought_again_90d"],
    "numerical": ["num_orders", "avg_basket_value", "days_since_last_order"],
}
EXPERIMENT_DIR = pathlib.Path("experiment")
EXPERIMENT_DIR.mkdir(exist_ok=True)

## Try it: what happens with the platform's defaults

Run the cell below as-is, with the platform's default settings. It will fail. Read the error before moving to the next cell -- figuring out *why* it fails is the actual point of this section.

In [ ]:
try:
    pre = preprocessing.Preprocessor(
        dataframe=working_table.copy(),
        experiment_directory=EXPERIMENT_DIR,
        column_metadata=column_metadata,
        # remove_numerical_outliers=True is the platform's own default -- not set explicitly here on purpose.
    )
    pre.output_dataframe
except Exception as e:
    print(f"{type(e).__name__}: {e}")

### Why: a percentile-based filter breaks on a column that's mostly one number

The platform's default outlier removal trims every numerical column to its 0.5th-99.5th percentile range, then drops anything outside it, using a filter that requires a value to be *strictly greater than* the lower cut and *strictly less than* the upper one. That's a sound approach for `avg_basket_value`, where the extreme values are genuine outliers. It breaks completely on `num_orders`, because in this table, over 90% of customers placed exactly one order before the cutoff -- so both the 0.5th and the 99.5th percentile of that column land on the same number, `1`. A strict filter of `(value > 1) and (value < 1)` is impossible for any value at all, including 1 itself, so the entire table gets dropped, and every downstream step fails on an empty dataframe.

This isn't a hypothetical reading of the source. It's what actually happened when this notebook was built, and it's the reason the five questions from the pipeline-risk-points lesson apply here too: reading what a check actually does, rather than trusting its name, is what surfaces this before it becomes a silent, empty dataframe with no error at all in a slightly less skewed dataset.

The fix: turn the platform's automatic outlier removal off, and apply it by hand only to the one column where it's actually appropriate.

In [ ]:
pre = preprocessing.Preprocessor(
    dataframe=working_table.copy(),
    experiment_directory=EXPERIMENT_DIR,
    column_metadata=column_metadata,
    preprocess_metadata=dict(
        remove_duplicates=True,
        preprocess_missing_values=True,
        missing_values_preprocessing_method="drop",
        remove_numerical_outliers=False,
    ),
)
preprocessed = pre.output_dataframe

before = len(preprocessed)
preprocessed = numerical_variables_processing.remove_column_outliers(
    dataframe=preprocessed, column_name="avg_basket_value"
)
print(f"rows after preprocessing + manual outlier trim: {len(preprocessed)} (dropped {before - len(preprocessed)} for basket-value outliers)")
print("positive rate after preprocessing:", preprocessed["bought_again_90d"].mean())

## The logs, and why they exist

Every check above wrote to a log file, in a workspace folder, not just to this notebook's output. That's deliberate: the whole point of a shareable log is that someone who cannot see the raw table can still see what the preprocessing step found, and help debug a model trained on it, without ever touching the data itself.

Read the actual log below. It should show a missing-value warning for `top_category` (some orders have no matched product), a cardinality note on `region` and `primary_payment_type` (both well under the platform's recommended 50-100 distinct values), and class-imbalance warnings, including one on `bought_again_90d` itself, at the "moderate" level this session's lessons describe.

In [ ]:
log_path = sorted(EXPERIMENT_DIR.rglob("*.log"))[0]
print(log_path.read_text())

> **Read a log before sending it.** The warnings above name the actual columns and category values that triggered them, which means a "safe to share" log can still carry your column names and some of your category labels. That's a real, if minor, disclosure -- decide whether it matters for a given dataset before it goes external.

---

## Training and auto-tuning a synthetic data model

`Preprocessor` cleaned the table. This section trains a generative model on it and searches for good hyperparameters automatically, rather than hand-tuning them.

A `synthcity` data loader needs to know which column is the prediction target, since several of its evaluation metrics (including the suitability check later in this notebook) train a model on the data and need to know what to predict.

In [ ]:
data_loader = dataloader.GenericDataLoader(
    preprocessed, target_column="bought_again_90d", train_size=0.75
)

# One metric per family named in the lessons: a sanity check, a statistical-fidelity
# check, a suitability/performance check, and a privacy check. All four are defined
# so that higher is better, which is what lets the search maximise their mean.
EVALUATION_METRICS = {
    "sanity": ["close_values_probability"],
    "stats": ["inv_kl_divergence"],
    "performance": ["xgb"],
    "privacy": ["k-anonymization"],
}
NUMBER_OF_TRIALS = 5  # the platform's own docs warn this is slow: each trial is a full training run

### Try it: tune TVAE

This runs `NUMBER_OF_TRIALS` full training runs of a tabular variational autoencoder, each with a different sampled set of hyperparameters, and keeps the one that scores best on the four metrics above. Expect this to take several minutes on a CPU runtime -- this is the "time and resource consuming" warning the platform's own documentation gives, not an accident.

In [ ]:
tvae_workspace = pathlib.Path("experiment_tvae")
tvae_workspace.mkdir(exist_ok=True)

tvae_model = Plugins().get("tvae", workspace=tvae_workspace)
tvae_tuner = sdmt.SyntheticDataModelTuner(
    data_loader=data_loader,
    synthetic_data_model=tvae_model,
    task_type="classification",
    number_of_trials=NUMBER_OF_TRIALS,
    optimization_direction="maximize",
    evaluation_metrics=EVALUATION_METRICS,
    experiment_directory=tvae_workspace,
)
print("best TVAE hyperparameters:", tvae_tuner.best_hyperparameters)
tvae_tuner.best_synthetic_data_model_evaluation_report

### Try it: tune CTGAN

Same search, same four metrics, a different model. CTGAN trains two networks against each other (a generator and a discriminator) rather than one autoencoder, and its conditional generation is specifically meant to handle rare categorical values -- worth watching for in the `top_category` and `region` columns, both of which have a long tail of infrequent values.

In [ ]:
ctgan_workspace = pathlib.Path("experiment_ctgan")
ctgan_workspace.mkdir(exist_ok=True)

ctgan_model = Plugins().get("ctgan", workspace=ctgan_workspace)
ctgan_tuner = sdmt.SyntheticDataModelTuner(
    data_loader=data_loader,
    synthetic_data_model=ctgan_model,
    task_type="classification",
    number_of_trials=NUMBER_OF_TRIALS,
    optimization_direction="maximize",
    evaluation_metrics=EVALUATION_METRICS,
    experiment_directory=ctgan_workspace,
)
print("best CTGAN hyperparameters:", ctgan_tuner.best_hyperparameters)
ctgan_tuner.best_synthetic_data_model_evaluation_report

### What the search found

Two plots make the search itself inspectable, rather than a black box that just hands back a winner:

In [ ]:
tvae_tuner.display_parallel_hyperparameter_coordinates()

In [ ]:
tvae_tuner.display_hyperparameter_importances()

The parallel-coordinates plot draws one line per trial across every hyperparameter axis and the resulting score, so a band of lines converging in the same region of an axis shows where the good scores came from. The importances plot ranks each hyperparameter by how much varying it moved the score -- which knobs mattered for this dataset, and which could have been left at their defaults.

### Picking a winner and generating the synthetic table

Compare the two evaluation reports above on the metric that matters most for this week's use case, `performance.xgb` (suitability), and keep whichever model actually did better on the thing this dataset is for. Generate a full-size synthetic table from that model.

In [ ]:
# Compare on the suitability metric specifically -- the one this week's use case
# actually depends on -- rather than the search's blended objective.
tvae_suitability = tvae_tuner.best_synthetic_data_model_evaluation_report.loc["performance.xgb.mean"].iloc[0] \
    if "performance.xgb.mean" in tvae_tuner.best_synthetic_data_model_evaluation_report.index else None
ctgan_suitability = ctgan_tuner.best_synthetic_data_model_evaluation_report.loc["performance.xgb.mean"].iloc[0] \
    if "performance.xgb.mean" in ctgan_tuner.best_synthetic_data_model_evaluation_report.index else None
print("TVAE suitability:", tvae_suitability)
print("CTGAN suitability:", ctgan_suitability)

best_tuner = tvae_tuner if (tvae_suitability or 0) >= (ctgan_suitability or 0) else ctgan_tuner
best_name = "tvae" if best_tuner is tvae_tuner else "ctgan"
print("\nwinner:", best_name)

synthetic_table = sdmt.generate_synthetic_data_with_synthetic_data_model(
    count=len(preprocessed), model=best_tuner.best_synthetic_data_model
)
synthetic_table.to_csv("synthetic_customer_table.csv", index=False)
synthetic_table.head()

---

## Reading the evaluation report

The winning model's evaluation report already showed the four metrics the search optimised against. This section reads a fuller report, and then goes one step further: it actually trains the propensity model this whole exercise was for, once on the synthetic table and once on the real one, and compares them on real, held-out customers.

In [ ]:
full_report = best_tuner.best_synthetic_data_model_full_evaluation_report
full_report

Four families, four different questions:

- **Sanity** (`close_values_probability` and neighbours) asks whether the output resembles real data at all -- the check that catches a model that produced nonsense.
- **Statistical fidelity** (`inv_kl_divergence` and neighbours) asks how closely the synthetic table's distributions match the real one's, column by column and across relationships between columns.
- **Performance / suitability** (`xgb` and neighbours) asks the question this week's use case actually depends on: does a model trained on the synthetic table make accurate predictions on real customers?
- **Privacy** (`k-anonymization` and neighbours) asks how exposed an individual synthetic record is. Higher k means each synthetic customer looks identical, on the attributes checked, to at least k-1 others.

These fight each other by construction. A model that reproduced the real table exactly would score perfectly on fidelity and suitability and offer no privacy at all, because the output would be the input. A model that generated random values would score perfectly on privacy and be useless for anything else. Every usable model sits somewhere between those two, and the "right" point on that trade-off is a judgment about how this specific table will be used, not a number the platform can hand back on its own.

### Suitability, tested directly: does the propensity model actually work?

The evaluation report's `performance.xgb` metric already trains a gradient-boosted model on the synthetic data and tests it on real data. This section does the same thing explicitly, with the metrics Session 2 taught for exactly this kind of rare-outcome problem: precision, recall, and F1, not accuracy, since well under 10% of customers in this table bought again.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import xgboost as xgb

CAT_COLS = ["region", "primary_payment_type", "top_category"]
NUM_COLS = ["num_orders", "avg_basket_value", "days_since_last_order"]
TARGET = "bought_again_90d"

real_train, real_test = train_test_split(
    preprocessed, test_size=0.25, stratify=preprocessed[TARGET], random_state=42
)

def make_pipeline():
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ])
    # n_jobs=1: avoids a real threading deadlock seen between xgboost and torch
    # in the same process.
    clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, eval_metric="logloss", n_jobs=1)
    return Pipeline([("pre", pre), ("clf", clf)])

def evaluate(train_df, label):
    pipe = make_pipeline()
    pipe.fit(train_df[CAT_COLS + NUM_COLS], train_df[TARGET])
    preds = pipe.predict(real_test[CAT_COLS + NUM_COLS])
    print(
        f"{label:22s} accuracy={accuracy_score(real_test[TARGET], preds):.3f}  "
        f"precision={precision_score(real_test[TARGET], preds, zero_division=0):.3f}  "
        f"recall={recall_score(real_test[TARGET], preds, zero_division=0):.3f}  "
        f"f1={f1_score(real_test[TARGET], preds, zero_division=0):.3f}"
    )

evaluate(real_train, "trained on REAL")
evaluate(synthetic_table, "trained on SYNTHETIC")

Both numbers are tested on the same real, held-out customers -- only the training data changes. Notice the accuracy figure first, and then notice how little it says: with a positive rate this low, a model that predicts "no" for every customer already scores well above 90% accuracy while catching none of the customers who actually come back. Precision, recall, and F1 are what actually distinguish a model that learned something from one that didn't, which is the same rare-class lesson Session 2 taught, showing up here in a place accuracy alone would have hidden it completely.

What this comparison licenses is narrow, and it's worth being precise about the boundary. A close result here says this model, trained this way, on this target column, transfers from synthetic to real. It says nothing about a different target, a different model family, or an exploratory analysis somebody runs later on the same synthetic file -- exactly the limit this session's first lesson named, because CTGAN and TVAE are general-purpose models with no validity guarantee outside what was actually tested.

---

## What this does, and does not, protect

The synthetic table above is safe to hand to someone who can't see the real one, for the specific purpose it was measured against. It is not anonymous in some absolute sense, and its privacy protection is a property of this dataset and this configuration, not of the method in general.

What it protects: individual customers. A well-tuned model makes it hard to point at a synthetic row and recover a specific real person, and the k-anonymity check above is one way of measuring how hard.

What it does not protect: the organisation's own aggregate patterns. Conversion rate, average basket value, the shape of the payment-type mix, the seasonality in order volume -- all of that survives into the synthetic table by design, because matching those patterns is the entire point of the exercise. A synthetic dataset protects the people in the data more than it protects the business the data describes.

**Also in Session 4**: when to use synthetic data at all, how a synthetic dataset actually gets made, and what the platform's metrics are measuring and why they trade off against each other -- this notebook assumes all three.

Questions belong in the Circle community.